In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("spark-batch")\
    .master("local[*]")\
    .getOrCreate()

spark
    

In [2]:
df = spark.read.format('csv').option('header', 'True').option('inferSchema', 'True').load('transactions.csv')

In [3]:
df.show()

+--------------+-----------+----------+-------+----------------+---------+-------+
|transaction_id|customer_id|product_id| amount|transaction_date|   status|channel|
+--------------+-----------+----------+-------+----------------+---------+-------+
|             1|      14758|         1| 132.87|      2025-07-03|   failed| online|
|             2|      22558|      4849|1521.89|      2025-07-09|   failed| online|
|             3|      39735|      4118|  55.68|      2025-09-14|   failed| online|
|             4|      26158|      1713| 595.92|      2025-03-31|   failed| online|
|             5|      47247|      3067| 944.27|      2025-10-09|completed| online|
|             6|      47724|         1| 474.45|      2025-04-30|completed| online|
|             7|      38571|         1|1125.01|      2025-06-16|completed| online|
|             8|       3018|      2969| 890.03|      2025-12-17|   failed| online|
|             9|      36511|      2047|1575.96|      2025-12-09|   failed| online|
|   

# Remove failed transactions

In [4]:
df = df.filter(df.status != "failed")

In [5]:
df.show()

+--------------+-----------+----------+-------+----------------+---------+-------+
|transaction_id|customer_id|product_id| amount|transaction_date|   status|channel|
+--------------+-----------+----------+-------+----------------+---------+-------+
|             5|      47247|      3067| 944.27|      2025-10-09|completed| online|
|             6|      47724|         1| 474.45|      2025-04-30|completed| online|
|             7|      38571|         1|1125.01|      2025-06-16|completed| online|
|            10|      49246|      3228| 554.46|      2025-05-24|completed| online|
|            11|      33772|      2058|1482.18|      2025-06-03|completed| online|
|            12|      25874|         1| 237.08|      2025-05-09|completed| online|
|            13|      14792|       843|1170.41|      2025-06-27|completed| online|
|            15|      29086|      4631|1161.97|      2025-07-12|completed| online|
|            18|      31975|      1467| 422.15|      2025-01-14|completed| online|
|   

# Deduplicate records

In [6]:
type(df)

pyspark.sql.dataframe.DataFrame

In [7]:
df = df.dropDuplicates(["transaction_id"])

In [8]:
df.show()

+--------------+-----------+----------+-------+----------------+---------+-------+
|transaction_id|customer_id|product_id| amount|transaction_date|   status|channel|
+--------------+-----------+----------+-------+----------------+---------+-------+
|             5|      47247|      3067| 944.27|      2025-10-09|completed| online|
|             6|      47724|         1| 474.45|      2025-04-30|completed| online|
|            12|      25874|         1| 237.08|      2025-05-09|completed| online|
|            13|      14792|       843|1170.41|      2025-06-27|completed| online|
|            15|      29086|      4631|1161.97|      2025-07-12|completed| online|
|            20|      15664|       741|1916.34|      2025-02-25|completed| online|
|            22|      47399|      4468| 862.97|      2025-06-15|completed| online|
|            31|      19950|        60| 173.69|      2025-12-14|completed| online|
|            52|      30391|         1|1128.13|      2025-04-21|completed| online|
|   

In [10]:
df.write \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .parquet("data/processed/transactions_parquet/")
